In [25]:
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [26]:
from utils_pourRL import *



In [27]:
import importlib
import utils_pourRL

importlib.reload(utils_pourRL)

Affichage = utils_pourRL.Affichage


[<utils_pourRL.pique object at 0x000001C0562E0830>, <utils_pourRL.pique object at 0x000001C055747B10>, <utils_pourRL.carre object at 0x000001C0562E0980>, <utils_pourRL.pique object at 0x000001C059EBB6F0>, <utils_pourRL.carre object at 0x000001C0597C56D0>, <utils_pourRL.pique object at 0x000001C055D90C00>, <utils_pourRL.pique object at 0x000001C05A132A50>, <utils_pourRL.pique object at 0x000001C05A1F7110>, <utils_pourRL.carre object at 0x000001C059EBBCE0>, <utils_pourRL.pique object at 0x000001C055C86350>, <utils_pourRL.pique object at 0x000001C055DEFEE0>, <utils_pourRL.carre object at 0x000001C055D93CE0>, <utils_pourRL.pique object at 0x000001C05A123F50>, <utils_pourRL.plateforme object at 0x000001C059ECB4D0>, <utils_pourRL.pique object at 0x000001C055D994F0>, <utils_pourRL.pique object at 0x000001C05A3D9450>, <utils_pourRL.pique object at 0x000001C05A0023D0>, <utils_pourRL.pique object at 0x000001C05A2A6B30>, <utils_pourRL.plateforme object at 0x000001C05A3A0910>, <utils_pourRL.pique 

In [28]:
class DQN(nn.Module):

    def __init__(self, n_observation = 5):
        super().__init__()
        self.layer1 = nn.Linear(n_observation, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, 2)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = self.layer3(x)
        return x

In [29]:
policy_net = DQN()
target_net = DQN()

In [57]:
def choisir_action(state, epsilon):
    if random.random() < epsilon:
        return random.choice([0,1])
    state = torch.tensor(state, dtype = torch.float32)
    q_values = policy_net(state)
    return torch.argmax(q_values).item()

epsilon = 0.01
nb_episodes = 1000

In [31]:
print("test")

test


In [72]:
import time
import pygame
import utils_pourRL

def jouer(jeu):
    state = jeu.reset()
    affichage = utils_pourRL.Affichage(jeu)
    running = True

    policy_net.eval()

    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        if not running:
            break

        state_tensor = torch.tensor(state, dtype=torch.float32)

        with torch.no_grad():
            q_values = policy_net(state_tensor)
            action = torch.argmax(q_values).item()

        next_state, reward, done = jeu.step(action)

        affichage.dessiner()
        pygame.display.flip()

        if done:
            print("Fin de la partie. Score :", jeu.score)
            running = False
        else:
            state = next_state

        time.sleep(1 / 60)

    pygame.quit()

jouer(jeu)

 

Fin de la partie. Score : 778


In [71]:
criterion = nn.SmoothL1Loss()
memory = deque(maxlen=10000)
optimizer = optim.Adam(policy_net.parameters(), lr = 0.0001)
step_count = 0

for episode in range(nb_episodes):

    state = jeu.reset()
    done = False
    
    while not done:
        step_count += 1
        action = choisir_action(state, epsilon)

        next_state, reward, done = jeu.step(action)

        memory.append(
            (state, action, reward, next_state, done)
        )

        if len(memory) >= 64:
            batch = random.sample(memory, 64)
            states, actions, rewards, next_states, dones = zip(*batch)
            states = torch.tensor(states, dtype=torch.float32)
            actions = torch.tensor(actions, dtype=torch.long)
            rewards = torch.tensor(rewards, dtype=torch.float32)
            next_states = torch.tensor(next_states, dtype=torch.float32)
            dones = torch.tensor(dones, dtype=torch.float32)

            output = policy_net(states)
            q_sa = output.gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q_values = target_net(next_states)
                max_next_q = next_q_values.max(dim=1).values
                cible = rewards + 0.9 * (1 - dones) * max_next_q
            loss = criterion(cible, q_sa)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if step_count % 500 == 0:
            target_net.load_state_dict(policy_net.state_dict())


        state = next_state
    print(loss)

tensor(0.0074, grad_fn=<SmoothL1LossBackward0>)


KeyboardInterrupt: 